# Plot the results of subclone interaction term estimations (winners)

This file evaluates which subline wins for the estimations for $k$ and $m$ and generates Figures 3c.

## Prep

Load needed packages

In [ ]:
%matplotlib widget
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import math
import seaborn as sns
import os
from statsmodels.formula import api as smf
from estimator import Estimator
pd.options.mode.chained_assignment = None

Close any figures generated from previous runs

In [ ]:
plt.close("all")

## Define data and user-input parameters

Define the file path for the configuration file (which contains the path to the growth data and estimated growth rates of the subclones), the result path, save path, and names of nude groups.

In [ ]:
groups = ["Grp. B2 nude (80% C1; 20% C11)", "Grp. B3 nude (50% C1; 50% C11)", "Grp. B4 nude (20% C1; 80% C11)"]
result_path = "results/subclone_winners/"
config = "config_subclone.json"
save_file = "figures/km_sign.svg" # Set to None if you don't want to save

Create an Estimator object from the config file

In [ ]:
es = Estimator(config)

Function to get the initial ratio of the subclones based on the group name

In [ ]:
def get_init(group):
    if "A1" in group or "B1" in group: return [1, 0]
    if "A2" in group or "B2" in group: return [0.8, 0.2]
    if "A3" in group or "B3" in group: return [0.5, 0.5]
    if "A4" in group or "B4" in group: return [0.2, 0.8]
    if "A5" in group or "B5" in group: return [0, 1]

## Load results

Loop through all files in path and create a data frame with those results

In [ ]:
results = []
for fname in [f for f in os.listdir(result_path) if ".csv" in f]:
    temp = pd.read_csv("{}/{}".format(result_path, fname))
    results += [temp]
results = pd.concat(results, ignore_index=True).drop_duplicates().reset_index()
results

## Plot

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(10, 3))
# Define color palette
colors = sns.color_palette("blend:black,grey", as_cmap=True)
# Loop through the groups and plot on subplots
i = 0
for group in results["group"].unique():
    temp = results[results["group"] == group]
    temp = temp.groupby(["k", "m"])["winner"].mean().reset_index()
    temp = pd.pivot(temp, index="k", columns="m", values="winner")    
    temp = temp.sort_index(ascending=False)
    sns.heatmap(temp, ax=axes[i], square=True, cmap=colors, cbar=False) # If you have an uneven number of k and m values, you might need to switch square to False
    axes[i].set_title(group)
    i += 1
plt.tight_layout()
if save_file:
    plt.savefig(save_file)
plt.show()